# Notebook 4: Approccio 3 - Word Embeddings

In questo notebook esploreremo come rappresentare le parole come vettori densi che catturano il significato semantico.

## Setup Iniziale

In [ ]:
# Installazione delle librerie necessarie
import subprocess
import sys

# Installa gensim se necessario
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gensim"])

print("Librerie installate con successo!")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from gensim.models import Word2Vec
import gensim.downloader as api
from itertools import combinations

# Impostazioni per la visualizzazione
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Setup completato!")

## 4.1 Dalla rappresentazione sparsa ai vettori densi

Nel precedente notebook abbiamo visto che le rappresentazioni "Bag of Words" e "one-hot encoding" sono:
- **Sparse**: contengono principalmente zeri
- **Altamente dimensionali**: una dimensione per ogni parola nel vocabolario
- **Senza semantica**: due parole diverse hanno vettori ortogonali, indipendentemente dalla loro somiglianza semantica

L'obiettivo dei **word embeddings** è creare vettori:
- **Densi**: con pochi zeri, valori significativi in poche dimensioni
- **Bassi-dimensionali**: tipicamente 50-300 dimensioni
- **Semantici**: parole simili hanno vettori simili

In [ ]:
# Esempio: Rappresentazione One-Hot vs Embedding

# Supponete un vocabolario di 10000 parole
vocab_size = 10000
embedding_dim = 100

# One-Hot: vettore con un solo 1 e 9999 zeri
one_hot_vector = np.zeros(vocab_size)
one_hot_vector[42] = 1  # Parola all'indice 42

# Embedding: 100 valori non-zero casuali
embedding_vector = np.random.randn(embedding_dim)

print("=== ONE-HOT ENCODING ===")
print(f"Dimensionalità: {vocab_size}")
print(f"Numero di zeri: {np.sum(one_hot_vector == 0)}")
print(f"Densità (non-zeri): {np.sum(one_hot_vector != 0) / vocab_size * 100:.4f}%")
print(f"Primi 50 elementi: {one_hot_vector[:50]}")

print("\n=== EMBEDDING ===")
print(f"Dimensionalità: {embedding_dim}")
print(f"Numero di zeri: {np.sum(embedding_vector == 0)}")
print(f"Densità (non-zeri): {np.sum(embedding_vector != 0) / embedding_dim * 100:.2f}%")
print(f"Primi 20 valori: {embedding_vector[:20]}")

# Calcolare similarità tra due vettori
one_hot_2 = np.zeros(vocab_size)
one_hot_2[43] = 1  # Un'altra parola
embedding_2 = np.random.randn(embedding_dim)

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

print("\n=== SIMILARITÀ ===")
print(f"Similarità One-Hot tra due parole diverse: {cosine_similarity(one_hot_vector, one_hot_2):.4f}")
print(f"Similarità Embedding (random): {cosine_similarity(embedding_vector, embedding_2):.4f}")
print("\nNota: One-Hot produce sempre 0 (vettori ortogonali)")
print("Gli embeddings possono catturare somiglianze semantiche")

## 4.2 L'intuizione: la geometria degli embeddings

Un'ottima intuizione viene dall'analogia delle **mele**:
Immaginate di descrivere le mele usando tre proprietà:
- **Colore** (scala 0-10: rosso vs verde)
- **Altezza** (scala 0-10: piccola vs grande)
- **Perimetro** (scala 0-10: sottile vs tondo)

Ogni mela può essere rappresentata come un punto nello spazio 3D. **Mele simili saranno vicine nello spazio!**

Lo stesso concetto vale per le parole: parole simili dovrebbero stare vicine nello spazio vettoriale.

In [ ]:
# Creare un dataset di mele con proprietà

apples_data = {
    'Nome': ['Gala', 'Fuji', 'Braeburn', 'Granny_Smith', 'Pink_Lady',
             'Golden_Delicious', 'Honeycrisp', 'Cortland', 'Mcintosh', 'Red_Delicious'],
    'Colore': [8, 9, 7, 2, 6, 7, 8, 5, 8, 9],  # 0=verde, 10=rosso
    'Altezza': [6, 7, 5, 5, 6, 6, 7, 5, 6, 5],  # 0=piccola, 10=grande
    'Perimetro': [7, 8, 6, 5, 7, 7, 8, 6, 7, 6]  # 0=sottile, 10=tondo
}

df_apples = pd.DataFrame(apples_data)
print("Dataset di Mele:")
print(df_apples)

# Visualizzazione 3D
fig = plt.figure(figsize=(12, 5))

# Sottotrama 1: 3D
ax1 = fig.add_subplot(121, projection='3d')
scatter = ax1.scatter(df_apples['Colore'], df_apples['Altezza'], df_apples['Perimetro'],
                       c=df_apples['Colore'], cmap='RdYlGn_r', s=100, alpha=0.6)

for idx, row in df_apples.iterrows():
    ax1.text(row['Colore'], row['Altezza'], row['Perimetro'], 
             row['Nome'], fontsize=8)

ax1.set_xlabel('Colore (0=verde, 10=rosso)')
ax1.set_ylabel('Altezza')
ax1.set_zlabel('Perimetro')
ax1.set_title('Mele nello spazio 3D')

# Sottotrama 2: 2D (Colore vs Altezza)
ax2 = fig.add_subplot(122)
scatter2 = ax2.scatter(df_apples['Colore'], df_apples['Altezza'],
                        c=df_apples['Perimetro'], cmap='viridis', s=100, alpha=0.6)

for idx, row in df_apples.iterrows():
    ax2.annotate(row['Nome'], (row['Colore'], row['Altezza']),
                fontsize=8, ha='center')

ax2.set_xlabel('Colore (0=verde, 10=rosso)')
ax2.set_ylabel('Altezza')
ax2.set_title('Mele: Colore vs Altezza')
plt.colorbar(scatter2, ax=ax2, label='Perimetro')

plt.tight_layout()
plt.show()

print("\nOsservazione: Mele rosse (Gala, Fuji, Red_Delicious) sono vicine tra loro!")
print("Mele verdi (Granny_Smith) è isolata.")
print("\nStesso concetto per le parole: parole con significati simili staranno vicine nello spazio vettoriale!")

## 4.3 L'ipotesi distribuzionale

**"You shall know a word by the company it keeps"** (J. R. Firth, 1957)

L'idea è semplice: **il significato di una parola è determinato dalle parole che la circondano (contesto)**.

Considerate la parola "oodles" - se non la conoscete, potete comunque capirne il significato dal contesto:
- "I have oodles of time" → grande quantità di tempo
- "oodles of money" → grande quantità di denaro

Allo stesso modo, parole come "car", "truck", "motorcycle", "van", "bus" appaiono spesso negli stessi contesti:
- "The ____ is parked"
- "The driver of the ____"
- "I drive a ____"

Perciò, queste parole hanno significati simili e dovrebbero avere embeddings simili.

In [ ]:
# Esempio: contesti per diverse parole

corpus = [
    "The car is parked outside the house",
    "The truck is large and powerful",
    "The driver of the car is careful",
    "The truck driver is experienced",
    "The motorcycle is fast and agile",
    "The van can carry many passengers",
    "The bus is full of people",
    "The driver of the bus is careful",
    "The car has four wheels",
    "The truck has a big engine"
]

print("=== CONTESTI PER PAROLE SIMILI ===")
print("\nParole di veicoli: car, truck, motorcycle, van, bus")
print("\nContesti comuni:")
print("- 'The ____ is ...'")
print("- 'The driver of the ____'")
print("- 'The ____ has ...'")

print("\nOsservazione: parole che appaiono in contesti simili hanno significati simili!")
print("\nEsempi dal corpus:")
for sentence in corpus:
    print(f"  {sentence}")

## 4.4 Word2Vec: CBOW e Skip-gram

**Word2Vec** è il modello più popolare per generare word embeddings. Esistono due architetture:

### CBOW (Continuous Bag of Words)
- **Input**: parole del contesto (finestra di parole attorno alla parola target)
- **Output**: la parola target (parola centrale)
- Idea: predire la parola dal contesto

### Skip-gram
- **Input**: la parola target (parola centrale)
- **Output**: parole del contesto
- Idea: predire il contesto dalla parola

Entrambi producono come effetto collaterale degli **embedding vettoriali** (i pesi della rete neurale nascosta).

In [ ]:
# Addestrare un modello Word2Vec semplice

# Corpus di esempio
sentences = [
    ["dog", "is", "a", "loyal", "pet"],
    ["cat", "is", "a", "cute", "pet"],
    ["dog", "and", "cat", "are", "animals"],
    ["dog", "barks", "loudly"],
    ["cat", "meows", "quietly"],
    ["dog", "plays", "with", "ball"],
    ["cat", "plays", "with", "toy"],
    ["pet", "is", "important"],
    ["dog", "is", "intelligent"],
    ["cat", "is", "independent"]
]

# Addestrare modello CBOW
print("Addestramento modello Word2Vec (CBOW)...")
model_cbow = Word2Vec(sentences=sentences, vector_size=50, window=3, min_count=1, sg=0, epochs=100)
print(f"Vocabolario: {len(model_cbow.wv)}")
print(f"Dimensione embedding: {model_cbow.vector_size}")

# Addestrare modello Skip-gram
print("\nAddestramento modello Word2Vec (Skip-gram)...")
model_skipgram = Word2Vec(sentences=sentences, vector_size=50, window=3, min_count=1, sg=1, epochs=100)

# Mostrare embeddings di alcune parole
print("\n=== EMBEDDINGS (CBOW) ===")
for word in ['dog', 'cat', 'pet', 'animal']:
    if word in model_cbow.wv:
        vec = model_cbow.wv[word]
        print(f"\n'{word}': {vec[:10]}... (prime 10 dimensioni su 50)")

## 4.5 Esplorare gli embeddings: similarità tra parole

Una volta addestrato il modello, possiamo usare gli embeddings per trovare parole simili utilizzando la **somiglianza del coseno**.

In [ ]:
# Caricare modello pre-addestrato di gensim (più grande, migliore)
print("Caricamento modello pre-addestrato GloVe...")
print("(La prima volta richiede il download)")

try:
    wv = api.load('glove-wiki-gigaword-50')
    print("Modello caricato con successo!")
except:
    print("Usando il modello Word2Vec addestrato localmente...")
    wv = model_skipgram.wv

# Funzione per trovare parole simili
def find_similar_words(word, topn=5):
    try:
        similar = wv.most_similar(word, topn=topn)
        print(f"\nParole simili a '{word}':")
        for similar_word, score in similar:
            print(f"  {similar_word:15s} {score:.4f}")
    except KeyError:
        print(f"Parola '{word}' non nel vocabolario")

# Testare similarità
test_words = ['computer', 'apple', 'car', 'dog']

print("=== SIMILARITÀ TRA PAROLE ===")
for word in test_words:
    find_similar_words(word, topn=5)

# Calcolare similarità tra coppie di parole
print("\n=== SIMILARITÀ TRA COPPIE ===")
word_pairs = [('computer', 'software'), ('dog', 'cat'), ('king', 'queen'), ('car', 'truck')]

for word1, word2 in word_pairs:
    try:
        sim = wv.similarity(word1, word2)
        print(f"Similarità '{word1}' - '{word2}': {sim:.4f}")
    except KeyError as e:
        print(f"Una delle parole non è nel vocabolario: {e}")

## 4.6 Analogie: la matematica degli embeddings

Una delle proprietà più interessanti degli embeddings è che **le relazioni semantiche sono preservate come relazioni geometriche**.

L'analogia classica: **king - man + woman = queen**

Matematicamente: `embedding('king') - embedding('man') + embedding('woman') ≈ embedding('queen')`

Questo funziona perché gli embeddings catturano relazioni come:
- **Genere**: la differenza tra "man" e "woman" è simile in molti contesti
- **Regalità**: la differenza tra "king" e "queen" è legata al genere

Quindi: `king` + (woman - man) = una parola che è a "reale" come king ma di genere femminile = queen!

In [ ]:
# Eseguire analogie

def solve_analogy(word1, word2, word3, topn=5):
    """Risolvi: word1 is to word2 as word3 is to ?"""
    try:
        results = wv.most_similar(positive=[word3, word2], negative=[word1], topn=topn)
        print(f"\nAnalogía: {word1} : {word2} :: {word3} : ?")
        print("Candidati:")
        for word, score in results:
            print(f"  {word:20s} {score:.4f}")
        return results[0][0] if results else None
    except KeyError as e:
        print(f"Parola non trovata: {e}")
        return None

print("=== ANALOGIE SEMANTICHE ===")

# Analogie classiche
analogies = [
    ("man", "woman", "king"),  # king:man :: queen:woman -> queen
    ("france", "paris", "italy"),  # italy:france :: ? : paris -> rome
    ("good", "better", "bad"),  # bad:good :: worse:better -> worse
    ("walk", "walked", "swim"),  # swim:walk :: swam:walked -> swam
]

results_analogies = []
for word1, word2, word3 in analogies:
    result = solve_analogy(word1, word2, word3, topn=3)
    if result:
        results_analogies.append((word1, word2, word3, result))

In [ ]:
# Visualizzare l'analogia king-queen con PCA

try:
    words_to_plot = ['man', 'woman', 'king', 'queen', 'prince', 'princess']
    vectors = np.array([wv[word] for word in words_to_plot])
    
    # Ridurre a 2D con PCA
    pca = PCA(n_components=2)
    vectors_2d = pca.fit_transform(vectors)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plottare i punti
    scatter = ax.scatter(vectors_2d[:, 0], vectors_2d[:, 1], s=200, alpha=0.6, c=range(len(words_to_plot)), cmap='tab10')
    
    # Aggiungere etichette
    for i, word in enumerate(words_to_plot):
        ax.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]), 
                   fontsize=12, ha='center', fontweight='bold')
    
    # Disegnare l'analogia king -> queen
    king_idx = words_to_plot.index('king')
    queen_idx = words_to_plot.index('queen')
    man_idx = words_to_plot.index('man')
    woman_idx = words_to_plot.index('woman')
    
    # Freccia: king + (woman - man)
    ax.arrow(vectors_2d[king_idx, 0], vectors_2d[king_idx, 1],
            vectors_2d[woman_idx, 0] - vectors_2d[man_idx, 0],
            vectors_2d[woman_idx, 1] - vectors_2d[man_idx, 1],
            head_width=0.15, head_length=0.1, fc='red', ec='red', alpha=0.6, linewidth=2)
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_title('Analogia: king - man + woman ≈ queen')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Varianza spiegata: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}")
except:
    print("Alcune parole non sono nel vocabolario")

## 4.7 Clustering con Word Embeddings

Possiamo usare i word embeddings per clustering semantico:
1. Ottenere gli embeddings di un insieme di parole
2. Applicare un algoritmo di clustering (es. K-Means)
3. Visualizzare con t-SNE

Le parole semanticamente simili dovrebbero trovarsi nello stesso cluster.

In [ ]:
# Clustering di parole da diverse categorie

words_to_cluster = [
    # Frutti
    'apple', 'banana', 'orange', 'grape', 'strawberry',
    # Veicoli
    'car', 'truck', 'bus', 'bicycle', 'motorcycle',
    # Paesi
    'italy', 'france', 'germany', 'spain', 'greece',
    # Sport
    'football', 'basketball', 'tennis', 'swimming', 'hockey'
]

# Ottenere embeddings
print(f"Recupero embeddings per {len(words_to_cluster)} parole...")
vectors = []
valid_words = []

for word in words_to_cluster:
    try:
        vectors.append(wv[word])
        valid_words.append(word)
    except KeyError:
        print(f"Parola '{word}' non trovata")

vectors = np.array(vectors)

print(f"Embeddings ottenuti: {len(valid_words)} su {len(words_to_cluster)}")

# Clustering con K-Means
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(vectors)

print(f"\nCluster assegnati:")
for i, (word, label) in enumerate(zip(valid_words, cluster_labels)):
    print(f"  {word:15s} -> Cluster {label}")

In [ ]:
# Visualizzazione con t-SNE

print("Esecuzione t-SNE (può richiedere alcuni secondi)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(valid_words)-1))
vectors_2d = tsne.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(12, 8))

# Colori per cluster
colors = ['red', 'blue', 'green', 'orange']
category_colors = {'apple': 'red', 'banana': 'red', 'orange': 'red', 'grape': 'red', 'strawberry': 'red',
                   'car': 'blue', 'truck': 'blue', 'bus': 'blue', 'bicycle': 'blue', 'motorcycle': 'blue',
                   'italy': 'green', 'france': 'green', 'germany': 'green', 'spain': 'green', 'greece': 'green',
                   'football': 'orange', 'basketball': 'orange', 'tennis': 'orange', 'swimming': 'orange', 'hockey': 'orange'}

for i, (x, y) in enumerate(vectors_2d):
    word = valid_words[i]
    color = category_colors.get(word, 'gray')
    ax.scatter(x, y, s=300, alpha=0.6, c=color)
    ax.annotate(word, (x, y), fontsize=10, ha='center', va='center', fontweight='bold')

# Legenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label='Frutti'),
    Patch(facecolor='blue', label='Veicoli'),
    Patch(facecolor='green', label='Paesi'),
    Patch(facecolor='orange', label='Sport')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)

ax.set_xlabel('t-SNE Dimensione 1')
ax.set_ylabel('t-SNE Dimensione 2')
ax.set_title('Clustering di Parole con Word Embeddings (t-SNE visualization)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nOsservazione: Parole della stessa categoria dovrebbero essere vicine!")

## 4.8 GloVe e fastText

Word2Vec è il più popolare, ma esistono altri approcci:

### GloVe (Global Vectors)
- Combina le intuizioni di Word2Vec con le statistiche di co-occorrenza globali
- Migliore per catturare similarità globali
- Spesso produce risultati leggermente migliori di Word2Vec

### fastText
- Basato su Word2Vec ma rappresenta parole come somme di n-grami di caratteri
- Vantaggio: può generare embeddings per parole non viste (OOV - Out of Vocabulary)
- Migliore per lingue con vocabolario grande o parole rare

In [ ]:
# Confronto tra Word2Vec e GloVe

print("=== CONFRONTO Word2Vec vs GloVe ===")
print("\nWord2Vec (Skip-gram, addestrato localmente):")
print(f"Vocabolario: {len(model_skipgram.wv)}")
print(f"Dimensioni embedding: {model_skipgram.vector_size}")

print("\nGloVe (pre-addestrato su Wikipedia):")
print(f"Vocabolario: ~400,000 parole")
print(f"Dimensioni embedding: 50")

# Confrontare i risultati su analogie e similarità
test_word = 'computer'

print(f"\nParole simili a '{test_word}':")
print("\nWord2Vec:")
try:
    for word, score in model_skipgram.wv.most_similar(test_word, topn=3):
        print(f"  {word:15s} {score:.4f}")
except:
    print("  (Parola non nel vocabolario)")

print("\nGloVe:")
try:
    for word, score in wv.most_similar(test_word, topn=3):
        print(f"  {word:15s} {score:.4f}")
except:
    print("  (Parola non nel vocabolario)")

print("\n=== fastText ===")
print("fastText vantaggi:")
print("1. Gestisce parole OOV (non viste durante l'addestramento)")
print("2. Usa informazione subword (n-grami di caratteri)")
print("3. Migliore per lingue morfologicamente ricche")
print("\nEsempio: fastText potrebbe gestire 'unhappily' anche se non l'ha visto")
print("         generando un embedding dalla combinazione di n-grami")

## 4.9 Applicazioni

Gli embeddings hanno numerose applicazioni:
1. **Motori di ricerca**: trovare documenti simili
2. **Riconoscimento di contenuti offensivi**: classificare il tono
3. **Sistemi di raccomandazione**: suggerire prodotti simili
4. **Analisi del sentiment**: classificare emozioni
5. **Matching CV-lavoro**: trovare il candidato migliore per una posizione

Vediamo un semplice esempio di matching tra CV e posizioni lavorative.

In [ ]:
# Applicazione: Matching CV-Job Description

job_descriptions = [
    "We are looking for a software engineer with Python and machine learning expertise",
    "Marketing manager needed with experience in digital campaigns and social media",
    "Data analyst required with SQL, statistics and visualization skills"
]

cvs = [
    "I have 5 years of Python experience, worked with machine learning frameworks, scikit-learn and tensorflow",
    "Digital marketing professional with 3 years in social media management and campaign planning",
    "Data specialist with SQL expertise, statistical analysis and Tableau visualization experience"
]

# Funzione per convertire testo in embedding (media dei word embeddings)
def text_to_embedding(text):
    words = text.lower().split()
    embeddings = []
    for word in words:
        try:
            embeddings.append(wv[word])
        except KeyError:
            pass
    
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(wv.vector_size)

# Calcolare embeddings
job_embeddings = [text_to_embedding(job) for job in job_descriptions]
cv_embeddings = [text_to_embedding(cv) for cv in cvs]

# Calcolare similarità
print("=== MATCHING CV - JOB ===")
print("\nMatrice di similarità (coseno):")
print("\n" + " " * 25 + "Job 1" + " " * 5 + "Job 2" + " " * 5 + "Job 3")

similarity_matrix = np.zeros((len(cvs), len(job_descriptions)))

for i, cv_emb in enumerate(cv_embeddings):
    similarities = []
    for j, job_emb in enumerate(job_embeddings):
        sim = cosine_similarity(cv_emb, job_emb)
        similarities.append(sim)
        similarity_matrix[i, j] = sim
    
    best_job = np.argmax(similarities)
    print(f"CV {i+1:d}:  {similarities[0]:.4f}     {similarities[1]:.4f}     {similarities[2]:.4f}    --> Best: Job {best_job+1}")

print("\nDettagli:")
for i in range(len(cvs)):
    best_job_idx = np.argmax(similarity_matrix[i])
    print(f"\nCV {i+1}: {cvs[i][:50]}...")
    print(f"  Miglior match: Job {best_job_idx+1} (similarità: {similarity_matrix[i, best_job_idx]:.4f})")
    print(f"  {job_descriptions[best_job_idx][:70]}...")

---

# ESERCIZIO

In questa sezione dovrai applicare quanto imparato.

## Compiti:

**(a)** Carica gli embeddings pre-addestrati GloVe (glove-wiki-gigaword-50)

**(b)** Trova i 10 parole più simili a ciascuna di queste 5 parole:
   - economy
   - python
   - university
   - football
   - pizza

**(c)** Risolvi 5 task di analogia:
   1. king : man :: queen : ?
   2. paris : france :: rome : ?
   3. good : better :: bad : ?
   4. run : ran :: swim : ?
   5. small : smaller :: large : ?

**(d)** Prendi 20 parole da 4 categorie diverse:
   - 5 parole di animali
   - 5 parole di frutti
   - 5 parole di colori
   - 5 parole di mestieri
   
   Ottieni i loro embeddings, clusterizzali con K-Means (k=4), e visualizza con t-SNE

**(e)** Dati 3 descrizioni di lavoro e 3 CV:
   - Calcola la similarità coseno tra ogni coppia (job, CV)
   - Trova il miglior match per ogni job
   - Stampa i risultati in una matrice

Scrivi il tuo codice nella cella seguente.

In [ ]:
# SOLUZIONE ESERCIZIO
# Sostituisci il pass con il tuo codice

pass

---

# SOLUZIONE

In [ ]:
# (a) Caricamento embeddings GloVe
print("=" * 80)
print("SOLUZIONE ESERCIZIO - WORD EMBEDDINGS")
print("=" * 80)

print("\n(a) Caricamento embeddings GloVe...")
try:
    glove_vectors = api.load('glove-wiki-gigaword-50')
    print(f"✓ GloVe caricato con successo!")
    print(f"  Vocabolario: {len(glove_vectors)}")
    print(f"  Dimensioni: {glove_vectors.vector_size}")
except:
    print("⚠ Errore nel caricamento, usando Word2Vec locale")
    glove_vectors = wv

In [ ]:
# (b) Trovare le 10 parole più simili
print("\n(b) 10 parole più simili per ciascun termine:")
print("-" * 60)

target_words = ['economy', 'python', 'university', 'football', 'pizza']
similar_results = {}

for word in target_words:
    print(f"\nParola: '{word}'")
    try:
        similar = glove_vectors.most_similar(word, topn=10)
        similar_results[word] = similar
        for i, (sim_word, score) in enumerate(similar, 1):
            print(f"  {i:2d}. {sim_word:20s} (similarità: {score:.4f})")
    except KeyError:
        print(f"  ✗ Parola non trovata nel vocabolario")
        similar_results[word] = []

In [ ]:
# (c) Risolvere analogie
print("\n(c) Risoluzioni di analogie semantiche:")
print("-" * 60)

analogies = [
    ('man', 'woman', 'king', 'Genere reale'),
    ('france', 'paris', 'italy', 'Capitale di nazione'),
    ('good', 'better', 'bad', 'Comparativo'),
    ('ran', 'run', 'swam', 'Tempo verbale'),
    ('smaller', 'small', 'large', 'Diminutivo')
]

analogies_results = []

for word1, word2, word3, description in analogies:
    print(f"\n{description}")
    print(f"  Analogia: {word1} : {word2} :: {word3} : ?")
    
    try:
        results = glove_vectors.most_similar(positive=[word3, word2], negative=[word1], topn=5)
        answer = results[0][0]
        score = results[0][1]
        analogies_results.append((word1, word2, word3, answer, score))
        
        print(f"  → Risposta: {answer} (similarità: {score:.4f})")
        print(f"    Top 5 candidati:")
        for i, (w, s) in enumerate(results, 1):
            print(f"      {i}. {w:15s} ({s:.4f})")
    except KeyError as e:
        print(f"  ✗ Errore: parola non trovata ({e})")

In [ ]:
# (d) Clustering di 20 parole da 4 categorie
print("\n(d) Clustering di parole da 4 categorie:")
print("-" * 60)

# Definire le categorie
categories = {
    'Animali': ['dog', 'cat', 'elephant', 'lion', 'bird'],
    'Frutti': ['apple', 'banana', 'orange', 'grape', 'strawberry'],
    'Colori': ['red', 'blue', 'yellow', 'green', 'purple'],
    'Mestieri': ['doctor', 'teacher', 'engineer', 'chef', 'pilot']
}

# Raccogliere tutte le parole
all_words = []
word_to_category = {}

for category, words in categories.items():
    for word in words:
        all_words.append(word)
        word_to_category[word] = category

print(f"Parole totali: {len(all_words)}")
print(f"Categorie: {list(categories.keys())}")

# Ottenere embeddings
valid_words_cluster = []
vectors_cluster = []

for word in all_words:
    try:
        vectors_cluster.append(glove_vectors[word])
        valid_words_cluster.append(word)
    except KeyError:
        print(f"  Parola '{word}' non trovata")

vectors_cluster = np.array(vectors_cluster)

print(f"Embeddings ottenuti: {len(valid_words_cluster)} su {len(all_words)}")

# Clustering con K-Means
kmeans_cluster = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans_cluster.fit_predict(vectors_cluster)

print("\nAssegnazione dei cluster:")
for word, label in zip(valid_words_cluster, labels):
    category = word_to_category.get(word, 'Unknown')
    print(f"  {word:15s} → Cluster {label}  (Categoria reale: {category})")

In [ ]:
# Visualizzazione con t-SNE
print("\nGenerazione visualizzazione t-SNE...")

tsne_cluster = TSNE(n_components=2, random_state=42, perplexity=min(30, len(valid_words_cluster)-1))
vectors_2d_cluster = tsne_cluster.fit_transform(vectors_cluster)

fig, ax = plt.subplots(figsize=(14, 10))

# Colori per categorie reali
category_to_color = {
    'Animali': '#FF6B6B',
    'Frutti': '#FFD93D',
    'Colori': '#6BCB77',
    'Mestieri': '#4D96FF'
}

# Disegnare i punti
for i, (x, y) in enumerate(vectors_2d_cluster):
    word = valid_words_cluster[i]
    category = word_to_category.get(word, 'Unknown')
    color = category_to_color.get(category, 'gray')
    ax.scatter(x, y, s=500, alpha=0.7, c=color, edgecolors='black', linewidth=1.5)
    ax.annotate(word, (x, y), fontsize=10, ha='center', va='center', fontweight='bold')

# Legenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#FF6B6B', label='Animali'),
    Patch(facecolor='#FFD93D', label='Frutti'),
    Patch(facecolor='#6BCB77', label='Colori'),
    Patch(facecolor='#4D96FF', label='Mestieri')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=12, framealpha=0.9)

ax.set_xlabel('t-SNE Dimensione 1', fontsize=11)
ax.set_ylabel('t-SNE Dimensione 2', fontsize=11)
ax.set_title('Clustering con Word Embeddings (GloVe) - 20 parole da 4 categorie', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Visualizzazione completata")

In [ ]:
# (e) Matching tra job descriptions e CV
print("\n(e) Matching CV - Job Description:")
print("-" * 60)

job_descriptions_ex = [
    "Senior software engineer with expertise in Python, machine learning and data science",
    "Marketing specialist needed for digital campaigns and social media strategy",
    "Financial analyst required for investment portfolio and risk assessment"
]

cvs_ex = [
    "Python developer with 7 years experience in machine learning and artificial intelligence",
    "Social media manager with 4 years in digital marketing and campaign management",
    "Investment analyst with expertise in financial markets and portfolio management"
]

# Calcolare embeddings per job descriptions e CV
job_emb_ex = [text_to_embedding(job) for job in job_descriptions_ex]
cv_emb_ex = [text_to_embedding(cv) for cv in cvs_ex]

# Calcolare matrice di similarità
similarity_matrix_ex = np.zeros((len(cvs_ex), len(job_descriptions_ex)))

for i in range(len(cvs_ex)):
    for j in range(len(job_descriptions_ex)):
        similarity_matrix_ex[i, j] = cosine_similarity(cv_emb_ex[i], job_emb_ex[j])

print("\nMatrice di similarità (coseno):")
print("\n" + " " * 30 + "Job 1     Job 2     Job 3")
print("-" * 65)

for i in range(len(cvs_ex)):
    similarities = similarity_matrix_ex[i]
    best_job = np.argmax(similarities)
    print(f"CV {i+1}:  {similarities[0]:.4f}    {similarities[1]:.4f}    {similarities[2]:.4f}    --> Best: Job {best_job+1}")

print("\nDettagli matching:")
print("-" * 60)

for i in range(len(cvs_ex)):
    best_job_idx = np.argmax(similarity_matrix_ex[i])
    best_score = similarity_matrix_ex[i, best_job_idx]
    
    print(f"\nCV {i+1}:")
    print(f"  {cvs_ex[i][:70]}...")
    print(f"\n  Miglior match: JOB {best_job_idx+1} (similarità: {best_score:.4f})")
    print(f"  {job_descriptions_ex[best_job_idx][:70]}...")
    
    # Mostrare tutti gli score
    print(f"\n  Score per tutti i job:")
    for j in range(len(job_descriptions_ex)):
        score = similarity_matrix_ex[i, j]
        marker = "✓" if j == best_job_idx else " "
        print(f"    {marker} Job {j+1}: {score:.4f}")

In [ ]:
# Visualizzazione matrice di similarità
fig, ax = plt.subplots(figsize=(10, 6))

im = ax.imshow(similarity_matrix_ex, cmap='YlOrRd', aspect='auto')

# Etichette
ax.set_xticks(range(len(job_descriptions_ex)))
ax.set_yticks(range(len(cvs_ex)))
ax.set_xticklabels([f'Job {i+1}' for i in range(len(job_descriptions_ex))])
ax.set_yticklabels([f'CV {i+1}' for i in range(len(cvs_ex))])

# Aggiungere valori nelle celle
for i in range(len(cvs_ex)):
    for j in range(len(job_descriptions_ex)):
        text = ax.text(j, i, f'{similarity_matrix_ex[i, j]:.3f}',
                      ha="center", va="center", color="black", fontsize=12, fontweight='bold')

ax.set_title('Similarità Coseno: CV vs Job Descriptions', fontsize=13, fontweight='bold')
fig.colorbar(im, ax=ax, label='Similarità Coseno')

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("SOLUZIONE COMPLETATA")
print("=" * 80)